# LIBERO 재학습+재eval — **노드 5 전용** (노드 5대 × GPU 2, seed2 우선)

**이 노트북은 노드 5 몫만** (5개 = 노드 5대). 각 노드가 **자기 (모델,seed) 유닛을 학습+eval 끝까지**
책임 → 노드 간 대기 없음. **seed2 를 맨 앞 라운드로빈**으로 배치해 **신규 모델(mosaic·acm·acm2)이 든
seed2 가 가장 먼저 완성**된다 (baseline 3개는 이미 done).

- **유효 eval = overall n_ep ≥ 2500(=500×10)** + action + 체크포인트 ≥150k. 옛 50ep·미완·under-trained 은 무효.
- `bimamba_s7`=ours(acm2+carry+BiMamba+overlap), `mosaic`=carry+overlap → config 자동 매핑.
- 순서: **상태 → ① stale 제거 → ② 학습 → ③ eval**. 노드 5대에서 각각 `_node1`~`_node5` 열어 실행.
- ⚠️ 500ep eval 은 개당 ~40시간. 안 죽는 환경(nohup/tmux)에서.


In [ ]:
import sys, json, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
SEEDS  = [0, 1, 2, 3]
TARGET = cf.CKPT_STEP                     # 150,000 — 이보다 낮으면 재학습
N_TASKS = 10
MIN_VALID_EP = N_TASKS * cf.EVAL_N_EP // 2   # 유효 500ep = overall n_ep >= 2500 (=5000). 옛 50ep(500) 제외

# ── 우리 클러스터: 노드 5대 × GPU 2 (합 10) ──
GPU_COUNTS = [2, 2, 2, 2, 2]
NODE_IDX   = 4               # 이 노트북 = 노드 5 전용 (5개 중 5번째)

# folder_tag → 학습 config 키 (bimamba_s7=ours=acm2+carry+BiMamba+overlap, mosaic=carry+overlap)
MODELS  = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
TAG_CFG = {'act': 'act', 'acm': 'acm', 'acm2': 'acm2', 'bimamba': 'bimamba',
           'bimamba_s7': 'bimamba_mosaic', 'mosaic': 'mosaic_infer'}
for folder, cfgkey in TAG_CFG.items():
    cf.v23.MODEL_DIR_NAMES.setdefault(folder, folder)
    if folder not in cf.v23.MODEL_CONFIGS:
        cf.v23.MODEL_CONFIGS[folder] = cf.v23.MODEL_CONFIGS[cfgkey]

TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
print('목표 step:', f'{TARGET:,}', '| 유효 eval: overall n_ep >=', MIN_VALID_EP)
print('정책 매핑:', {k: cf.v23.MODEL_CONFIGS[k][0] for k in ['mosaic', 'bimamba_s7']})

## 상태 분류 — 재학습 / 재eval / stale 제거


In [ ]:
# ── 상태 분류: 재학습 / 재eval / stale제거 대상 ──
def tstep(t, s):
    return cf.v23.last_ckpt_step(TRAIN_ROOT / t / f'seed{s}')

def eval_rec(t, s):
    d = EVAL_ROOT / t / f'seed{s}'
    if not d.is_dir():
        return None
    best = None
    for info in d.rglob('eval_info.json'):
        try:
            ov = json.loads(info.read_text()).get('overall', {})
        except Exception:
            continue
        ne = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or ne > best['n_ep']:
            best = {'n_ep': ne, 'has_act': (info.parent / 'actions').is_dir(), 'info': info}
    return best

# 유닛(모델,seed) = 학습+eval 한 세트 → 노드가 자기 유닛을 끝까지(노드 간 대기 없음).
# seed2 를 맨 앞에 두고 라운드로빈 → seed2 유닛이 앞 노드들에 하나씩 = 가장 빨리 완성.
SEED_ORDER = [2, 1, 3, 0]        # ★seed2 우선 (baseline 3개 이미 done → 신규 모델만 채우면 완성)
N_NODES = len(GPU_COUNTS)
units = []                       # (t, s, step, needs_train, ev, valid)
for s in SEED_ORDER:
    for t in MODELS:
        step = tstep(t, s)
        nt = step is None or step < TARGET
        ev = eval_rec(t, s)
        ve = bool(ev and ev['n_ep'] >= MIN_VALID_EP and ev['has_act'] and not nt)
        if nt or not ve:          # 할 게 남은 유닛만
            units.append((t, s, step, nt, ev, ve))

mine = units[NODE_IDX::N_NODES]                                   # 라운드로빈 몫
mine_train  = [(t, s, TASK) for (t, s, step, nt, ev, ve) in mine if nt]
mine_eval   = [(t, s) for (t, s, step, nt, ev, ve) in mine]
rm_ckpts    = [TRAIN_ROOT / t / f'seed{s}' for (t, s, step, nt, ev, ve) in mine if nt and step is not None]
stale_infos = [ev['info'] for (t, s, step, nt, ev, ve) in mine if ev and not ve]

print(f'노드 {NODE_IDX} | 이 노드 유닛 {len(mine)}개 (전체 {len(units)}, 5노드 분배 {[len(units[i::N_NODES]) for i in range(N_NODES)]})')
print(f"{'model':<12}{'seed':>4}{'ckpt':>10}   할 일")
print('-' * 46)
for (t, s, step, nt, ev, ve) in mine:
    todo = (['학습'] if nt else []) + ['eval']
    print(f'{t:<12}{s:>4}{(f"{step:,}" if step else "-"):>10}   {" + ".join(todo)}' + ('   ★seed2' if s == 2 else ''))

s2_nodes = [i % N_NODES for i, u in enumerate(units) if u[1] == 2]
print(f'\nseed2 유닛 {sum(u[1] == 2 for u in units)}개 → 노드 {s2_nodes} 에 우선 분산')
print(f'이 노드: 학습 {len(mine_train)} / eval {len(mine_eval)}')

## ① stale 제거 (under-trained ckpt + 무효 eval_info) — dry-run → EXECUTE=True
**200k done·유효 500ep 은 안 건드림.**


In [ ]:
# ── ① stale 제거: under-trained 체크포인트 + 무효 eval_info ── 확인 후 EXECUTE=True ──
EXECUTE = False

print(f'{"제거" if EXECUTE else "DRY-RUN"} — under-trained ckpt {len(rm_ckpts)} + 무효 eval_info {len(stale_infos)}:')
for d in rm_ckpts:
    print(f'   CKPT {d.relative_to(cf.OUTPUT_BASE)}  (재학습 위해 제거)')
    if EXECUTE and d.is_dir():
        shutil.rmtree(d)
for inf in stale_infos:
    print(f'   INFO {inf.relative_to(EVAL_ROOT)}  (재eval 위해 제거 → skip 안 되게)')
    if EXECUTE and inf.exists():
        inf.unlink()
print('\n' + ('제거 완료.' if EXECUTE else '확인됐으면 EXECUTE=True. (이 노드 유닛만; done·유효 eval 은 안 건드림)'))

## ② 재학습 (목표 150k, 노드 5×2 GPU)


In [ ]:
# ── ② 이 노드 유닛 학습 (목표 150k) ── 학습 필요한 것만 (seed2 가 앞) ──
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX} | GPU {gpus} | 학습 {len(mine_train)}개: {[(t, s) for t, s, _ in mine_train]}')
if mine_train:
    cf.run_training_jobs(mine_train, gpus=gpus, prefetch_task=TASK)   # 목표 도달분 자동 skip, prefetch 포함
else:
    print('이 노드 학습할 것 없음 → ③ eval 로.')

## ③ 재eval (500ep, 노드 5×2 GPU) — ⚠️ 개당 ~40시간
②가 끝난 뒤 실행. 체크포인트 없는 건 자동 skip.


In [ ]:
# ── ③ 이 노드 유닛 eval (500ep) ── ②가 끝난 뒤 ── ⚠️ 개당 ~40시간(5000 에피소드) ──
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX} | GPU {gpus} | eval {len(mine_eval)}개: {mine_eval} × {cf.EVAL_N_EP}ep')
print('  체크포인트 없는 건 자동 skip(②에서 학습됨), 이미 유효한 건 skip. 안 죽는 환경(nohup)에서.')
if mine_eval:
    cf.run_libero_eval_jobs(mine_eval, gpus=gpus, n_episodes=cf.EVAL_N_EP)